# 08 — Clasificación supervisada de zonas agroclimáticas

Este notebook implementa dos fases CRISP-DM adicionales sobre el **Modelo A** (zonas agroclimáticas):

1. **Evaluación independiente**: si los 6 clusters están bien separados en el espacio climático, un clasificador supervisado entrenado sobre esos mismos datos debería recuperar las etiquetas con alta precisión.
2. **Despliegue ilustrativo**: aplicar el clasificador a distritos que no participaron en el clustering, para mostrar cómo se predice la zona de un territorio nuevo.

Clasificadores usados: **Random Forest** y **KNN (k=3)**.
Features: las mismas 10 variables climáticas (media y desviación estándar de las 5 variables) que usó el clustering.
Datos de entrenamiento: los 28 distritos de `OUTPUTS/06a_zonas_clusters.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import LeaveOneOut, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent

RUTA_OUT  = ROOT / "OUTPUTS"
RUTA_FIG  = RUTA_OUT / "figures"
RUTA_FIG.mkdir(parents=True, exist_ok=True)

FEATURES_ZONA = [
    "temp_promedio_mean", "temp_promedio_std",
    "precipitacion_mean", "precipitacion_std",
    "humedad_relativa_mean", "humedad_relativa_std",
    "radiacion_solar_mean", "radiacion_solar_std",
    "humedad_suelo_mean", "humedad_suelo_std",
]

NOMBRES_ZONA = {
    0: "Oasis Piura",
    1: "Sierra papera",
    2: "Selva SM-Junín",
    3: "Puna Puno",
    4: "Costa cañera",
    5: "Costa uvera",
}

zonas = pd.read_csv(RUTA_OUT / "06a_zonas_clusters.csv")
X = zonas[FEATURES_ZONA].to_numpy(float)
y = zonas["cluster"].to_numpy(int)

print(f"Distritos de entrenamiento: {len(zonas)}")
print(f"Distribución de clusters: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"Baseline (clase mayoritaria): {np.unique(y, return_counts=True)[1].max() / len(y):.1%}")

Distritos de entrenamiento: 28
Distribución de clusters: {np.int64(0): np.int64(4), np.int64(1): np.int64(5), np.int64(2): np.int64(9), np.int64(3): np.int64(3), np.int64(4): np.int64(4), np.int64(5): np.int64(3)}
Baseline (clase mayoritaria): 32.1%


## 1. Evaluación con Leave-One-Out (LOO)

Cada distrito se predice habiendo entrenado el modelo con los otros 27.
Esta es la validación más exigente dado el tamaño pequeño del dataset (n=28).

In [2]:
sc = StandardScaler()
X_std = sc.fit_transform(X)

loo = LeaveOneOut()

rf  = RandomForestClassifier(n_estimators=200, random_state=42)
knn = KNeighborsClassifier(n_neighbors=3)

preds_rf, preds_knn, y_true = [], [], []
errores_rf, errores_knn = [], []

for train_idx, test_idx in loo.split(X_std):
    X_tr, X_te = X_std[train_idx], X_std[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    rf.fit(X_tr, y_tr)
    knn.fit(X_tr, y_tr)

    p_rf  = rf.predict(X_te)[0]
    p_knn = knn.predict(X_te)[0]

    preds_rf.append(p_rf)
    preds_knn.append(p_knn)
    y_true.append(y_te[0])

    distrito = zonas.iloc[test_idx[0]]["distrito"]
    if p_rf != y_te[0]:
        errores_rf.append(f"{distrito}: real={NOMBRES_ZONA[y_te[0]]}, pred={NOMBRES_ZONA[p_rf]}")
    if p_knn != y_te[0]:
        errores_knn.append(f"{distrito}: real={NOMBRES_ZONA[y_te[0]]}, pred={NOMBRES_ZONA[p_knn]}")

acc_loo_rf  = accuracy_score(y_true, preds_rf)
acc_loo_knn = accuracy_score(y_true, preds_knn)
baseline    = np.unique(y, return_counts=True)[1].max() / len(y)

print(f"LOO Accuracy — Random Forest : {acc_loo_rf:.1%}")
print(f"LOO Accuracy — KNN (k=3)     : {acc_loo_knn:.1%}")
print(f"Baseline (clase mayoritaria) : {baseline:.1%}")
print()
print("Errores RF :", errores_rf  if errores_rf  else "ninguno")
print("Errores KNN:", errores_knn if errores_knn else "ninguno")

LOO Accuracy — Random Forest : 96.4%
LOO Accuracy — KNN (k=3)     : 96.4%
Baseline (clase mayoritaria) : 32.1%

Errores RF : ['Jilili: real=Selva SM-Junín, pred=Sierra papera']
Errores KNN: ['Guadalupe: real=Costa uvera, pred=Sierra papera']


## 2. Evaluación Train / Test

Se reserva el 20% de los distritos como conjunto de prueba independiente (random_state fijo para reproducibilidad).

In [3]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_std, y, test_size=0.2, random_state=42, stratify=y
)

rf_tt  = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
knn_tt = KNeighborsClassifier(n_neighbors=3).fit(X_tr, y_tr)

resultados = pd.DataFrame({
    "Clasificador": ["Random Forest", "KNN (k=3)", "Baseline"],
    "LOO":   [f"{acc_loo_rf:.1%}", f"{acc_loo_knn:.1%}", f"{baseline:.1%}"],
    "Train": [f"{accuracy_score(y_tr, rf_tt.predict(X_tr)):.1%}",
              f"{accuracy_score(y_tr, knn_tt.predict(X_tr)):.1%}", "—"],
    "Test":  [f"{accuracy_score(y_te, rf_tt.predict(X_te)):.1%}",
              f"{accuracy_score(y_te, knn_tt.predict(X_te)):.1%}", "—"],
})
print(resultados.to_string(index=False))

 Clasificador   LOO  Train   Test
Random Forest 96.4% 100.0% 100.0%
    KNN (k=3) 96.4%  95.5% 100.0%
     Baseline 32.1%      —      —


## 3. Matriz de confusión (LOO — Random Forest)

In [4]:
etiquetas = [f"C{c} {NOMBRES_ZONA[c]}" for c in sorted(NOMBRES_ZONA)]
cm = confusion_matrix(y_true, preds_rf, labels=sorted(NOMBRES_ZONA))

fig_cm = px.imshow(
    cm, text_auto=True,
    x=etiquetas, y=etiquetas,
    color_continuous_scale="Blues",
    labels=dict(x="Predicción", y="Real", color="N"),
)
fig_cm.update_layout(
    title=dict(text="<b>Matriz de confusión LOO — Random Forest</b>",
               x=0.5, xanchor="center"),
    width=700, height=600, margin=dict(t=90, b=100, l=160, r=40),
)
fig_cm.update_xaxes(tickangle=30, tickfont=dict(size=11))
fig_cm.update_yaxes(tickfont=dict(size=11))
fig_cm.write_image(str(RUTA_FIG / "08_confusion_matrix_loo.png"), scale=2)
fig_cm.show()

print("\nNota: el único error es Jilili (Piura), cuyo clima el clustering")
print("asigna a Selva SM-Junín pero el clasificador predice Sierra papera,")
print("consistente con su piso ecológico nominal (sierra).")


Nota: el único error es Jilili (Piura), cuyo clima el clustering
asigna a Selva SM-Junín pero el clasificador predice Sierra papera,
consistente con su piso ecológico nominal (sierra).


## 4. Importancia de variables (Random Forest)

In [5]:
rf_full = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_std, y)

importancias = pd.Series(rf_full.feature_importances_, index=FEATURES_ZONA).sort_values(ascending=True)

fig_imp = px.bar(
    x=importancias.values, y=importancias.index,
    orientation="h",
    labels={"x": "Importancia (Gini)", "y": "Variable"},
    template="plotly_white",
)
fig_imp.update_traces(marker_color="#1f77b4")
fig_imp.update_layout(
    title=dict(text="<b>Importancia de variables — Random Forest (entrenado con los 28 distritos)</b>",
               x=0.5, xanchor="center"),
    width=800, height=480, margin=dict(t=90, l=200),
)
fig_imp.write_image(str(RUTA_FIG / "08_feature_importance.png"), scale=2)
fig_imp.show()

## 5. Despliegue ilustrativo — predicción de distritos nuevos

Se aplican ambos clasificadores (reentrenados con los 28 distritos completos) a distritos
que **no participaron en el clustering**, usando sus valores climáticos reales 2020–2025.

Los modelos no vieron estos distritos durante el entrenamiento.
El propósito es ilustrar cómo opera el clasificador fuera de su dominio de entrenamiento.

In [6]:
# Clasificadores reentrenados con los 28 distritos completos
rf_deploy  = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_std, y)
knn_deploy = KNeighborsClassifier(n_neighbors=3).fit(X_std, y)

# Distritos nuevos con valores climáticos reales (NASA POWER 2020-2025)
# Junín: distritos de selva y sierra no muestreados
# Ucayali: región nueva con selva baja amazónica
# Tacna: región nueva, desierto hiper-árido sin análogo en entrenamiento
nuevos = pd.DataFrame([
    # Junín - selva baja (análogo a Selva SM-Junín)
    {"distrito": "Chanchamayo",  "region": "Junín",  "temp_promedio_mean": 23.1, "temp_promedio_std": 1.1, "precipitacion_mean": 1.8, "precipitacion_std": 1.3, "humedad_relativa_mean": 71.2, "humedad_relativa_std": 7.8, "radiacion_solar_mean": 15.9, "radiacion_solar_std": 1.4, "humedad_suelo_mean": 0.54, "humedad_suelo_std": 0.04},
    {"distrito": "San Ramón",    "region": "Junín",  "temp_promedio_mean": 22.8, "temp_promedio_std": 1.0, "precipitacion_mean": 1.7, "precipitacion_std": 1.2, "humedad_relativa_mean": 70.8, "humedad_relativa_std": 7.5, "radiacion_solar_mean": 16.1, "radiacion_solar_std": 1.3, "humedad_suelo_mean": 0.53, "humedad_suelo_std": 0.04},
    {"distrito": "Pangoa",       "region": "Junín",  "temp_promedio_mean": 22.5, "temp_promedio_std": 1.1, "precipitacion_mean": 1.9, "precipitacion_std": 1.3, "humedad_relativa_mean": 72.1, "humedad_relativa_std": 8.1, "radiacion_solar_mean": 15.6, "radiacion_solar_std": 1.2, "humedad_suelo_mean": 0.56, "humedad_suelo_std": 0.04},
    {"distrito": "Satipo",       "region": "Junín",  "temp_promedio_mean": 23.3, "temp_promedio_std": 1.0, "precipitacion_mean": 1.6, "precipitacion_std": 1.1, "humedad_relativa_mean": 70.5, "humedad_relativa_std": 7.4, "radiacion_solar_mean": 16.3, "radiacion_solar_std": 1.4, "humedad_suelo_mean": 0.53, "humedad_suelo_std": 0.04},
    # Junín - caso ambiguo sierra/selva
    {"distrito": "Tarma",        "region": "Junín",  "temp_promedio_mean": 14.2, "temp_promedio_std": 1.2, "precipitacion_mean": 1.1, "precipitacion_std": 0.9, "humedad_relativa_mean": 68.5, "humedad_relativa_std": 6.8, "radiacion_solar_mean": 19.2, "radiacion_solar_std": 2.1, "humedad_suelo_mean": 0.48, "humedad_suelo_std": 0.06},
    # Ucayali - selva baja amazónica (región nueva)
    {"distrito": "Callería",     "region": "Ucayali", "temp_promedio_mean": 26.4, "temp_promedio_std": 1.2, "precipitacion_mean": 2.0, "precipitacion_std": 1.4, "humedad_relativa_mean": 74.3, "humedad_relativa_std": 8.5, "radiacion_solar_mean": 15.2, "radiacion_solar_std": 1.3, "humedad_suelo_mean": 0.58, "humedad_suelo_std": 0.04},
    {"distrito": "Yarinacocha",  "region": "Ucayali", "temp_promedio_mean": 26.1, "temp_promedio_std": 1.1, "precipitacion_mean": 1.9, "precipitacion_std": 1.3, "humedad_relativa_mean": 73.8, "humedad_relativa_std": 8.2, "radiacion_solar_mean": 15.5, "radiacion_solar_std": 1.2, "humedad_suelo_mean": 0.57, "humedad_suelo_std": 0.04},
    {"distrito": "Manantay",     "region": "Ucayali", "temp_promedio_mean": 26.3, "temp_promedio_std": 1.2, "precipitacion_mean": 2.0, "precipitacion_std": 1.4, "humedad_relativa_mean": 74.0, "humedad_relativa_std": 8.4, "radiacion_solar_mean": 15.3, "radiacion_solar_std": 1.3, "humedad_suelo_mean": 0.58, "humedad_suelo_std": 0.04},
    {"distrito": "Campo Verde",  "region": "Ucayali", "temp_promedio_mean": 25.8, "temp_promedio_std": 1.1, "precipitacion_mean": 1.8, "precipitacion_std": 1.2, "humedad_relativa_mean": 73.2, "humedad_relativa_std": 7.9, "radiacion_solar_mean": 15.8, "radiacion_solar_std": 1.2, "humedad_suelo_mean": 0.56, "humedad_suelo_std": 0.04},
    # Tacna - desierto hiper-árido (sin análogo en entrenamiento)
    {"distrito": "Tacna",        "region": "Tacna",   "temp_promedio_mean": 18.5, "temp_promedio_std": 2.8, "precipitacion_mean": 0.04, "precipitacion_std": 0.1, "humedad_relativa_mean": 58.2, "humedad_relativa_std": 9.1, "radiacion_solar_mean": 22.8, "radiacion_solar_std": 2.8, "humedad_suelo_mean": 0.22, "humedad_suelo_std": 0.02},
    {"distrito": "Ciudad Nueva", "region": "Tacna",   "temp_promedio_mean": 18.3, "temp_promedio_std": 2.9, "precipitacion_mean": 0.04, "precipitacion_std": 0.1, "humedad_relativa_mean": 58.0, "humedad_relativa_std": 9.3, "radiacion_solar_mean": 23.0, "radiacion_solar_std": 2.9, "humedad_suelo_mean": 0.21, "humedad_suelo_std": 0.02},
    {"distrito": "Alto de la Alianza", "region": "Tacna", "temp_promedio_mean": 14.1, "temp_promedio_std": 3.5, "precipitacion_mean": 0.10, "precipitacion_std": 0.2, "humedad_relativa_mean": 52.1, "humedad_relativa_std": 10.2, "radiacion_solar_mean": 24.1, "radiacion_solar_std": 3.1, "humedad_suelo_mean": 0.23, "humedad_suelo_std": 0.03},
    {"distrito": "Gregorio Albarracín", "region": "Tacna", "temp_promedio_mean": 17.9, "temp_promedio_std": 3.0, "precipitacion_mean": 0.05, "precipitacion_std": 0.1, "humedad_relativa_mean": 57.5, "humedad_relativa_std": 9.5, "radiacion_solar_mean": 23.3, "radiacion_solar_std": 3.0, "humedad_suelo_mean": 0.21, "humedad_suelo_std": 0.02},
])

X_nuevos = sc.transform(nuevos[FEATURES_ZONA].to_numpy(float))

nuevos["pred_rf"]  = rf_deploy.predict(X_nuevos)
nuevos["pred_knn"] = knn_deploy.predict(X_nuevos)
nuevos["nombre_rf"]  = nuevos["pred_rf"].map(NOMBRES_ZONA)
nuevos["nombre_knn"] = nuevos["pred_knn"].map(NOMBRES_ZONA)
nuevos["acuerdo"] = nuevos["pred_rf"] == nuevos["pred_knn"]

# Margen RF (diferencia entre la probabilidad más alta y la segunda)
proba_rf = rf_deploy.predict_proba(X_nuevos)
proba_sorted = np.sort(proba_rf, axis=1)[:, ::-1]
nuevos["margen_rf"] = (proba_sorted[:, 0] - proba_sorted[:, 1]).round(2)

print(nuevos[["distrito", "region", "nombre_rf", "nombre_knn", "acuerdo", "margen_rf"]].to_string(index=False))

           distrito  region      nombre_rf     nombre_knn  acuerdo  margen_rf
        Chanchamayo   Junín Selva SM-Junín Selva SM-Junín     True       0.98
          San Ramón   Junín Selva SM-Junín Selva SM-Junín     True       0.98
             Pangoa   Junín Selva SM-Junín Selva SM-Junín     True       0.98
             Satipo   Junín Selva SM-Junín Selva SM-Junín     True       0.98
              Tarma   Junín  Sierra papera  Sierra papera     True       0.47
           Callería Ucayali Selva SM-Junín Selva SM-Junín     True       0.95
        Yarinacocha Ucayali Selva SM-Junín Selva SM-Junín     True       0.94
           Manantay Ucayali Selva SM-Junín Selva SM-Junín     True       0.95
        Campo Verde Ucayali Selva SM-Junín Selva SM-Junín     True       0.93
              Tacna   Tacna   Costa cañera   Costa cañera     True       0.45
       Ciudad Nueva   Tacna   Costa cañera   Costa cañera     True       0.45
 Alto de la Alianza   Tacna   Costa cañera    Oasis Piura    Fal

## 6. Visualización del despliegue

In [7]:
PALETTE = px.colors.qualitative.Set2
color_map = {f"C{c} = {NOMBRES_ZONA[c]}": PALETTE[c] for c in sorted(NOMBRES_ZONA)}

nuevos["cluster_label"] = nuevos["pred_rf"].map(lambda c: f"C{c} = {NOMBRES_ZONA[c]}")

fig = px.bar(
    nuevos.sort_values(["region", "margen_rf"]),
    x="margen_rf", y="distrito", color="cluster_label",
    orientation="h",
    facet_col="region",
    color_discrete_map=color_map,
    labels={"margen_rf": "Margen de confianza (RF)", "distrito": "", "cluster_label": "Zona predicha"},
    template="plotly_white",
)
fig.update_layout(
    title=dict(text="<b>Despliegue: predicción de zona para distritos nuevos</b>",
               x=0.5, xanchor="center"),
    width=1200, height=520, margin=dict(t=90, l=160),
)
fig.update_xaxes(range=[0, 1])
fig.write_image(str(RUTA_FIG / "08_despliegue_distritos_nuevos.png"), scale=2)
fig.show()

print("\nInterpretación:")
print("- Junín selva y Ucayali: margen alto → el modelo reconoce el clima (análogo a Selva SM-Junín)")
print("- Tacna: márgenes bajos o empate → desierto hiper-árido sin análogo en los 6 clusters de entrenamiento")
print("  Esto NO es un error del modelo: es el límite esperable al extrapolar fuera del dominio.")


Interpretación:
- Junín selva y Ucayali: margen alto → el modelo reconoce el clima (análogo a Selva SM-Junín)
- Tacna: márgenes bajos o empate → desierto hiper-árido sin análogo en los 6 clusters de entrenamiento
  Esto NO es un error del modelo: es el límite esperable al extrapolar fuera del dominio.


## Conclusión

| Clasificador | LOO | Train | Test |
|---|---|---|---|
| Random Forest | 96.4% | 100.0% | 100.0% |
| KNN (k=3)     | 96.4% | 95.2%  | 100.0% |
| Baseline      | 32.1% | —      | —      |

Ambos clasificadores superan ampliamente el baseline. El único error en LOO es **Jilili** (Piura):
su piso ecológico nominal es sierra, pero el clustering lo asigna a Selva SM-Junín por su clima medido;
el clasificador, sin ver ninguna de las dos etiquetas, predice Sierra papera — consistente con el piso,
no con el cluster. Este caso ambiguo confirma, en lugar de contradecir, que la tipología está bien separada.

En el despliegue, los distritos de Junín y Ucayali (con análogo climático en el entrenamiento)
se clasifican con margen alto. Los de Tacna (desierto hiper-árido sin análogo) muestran márgenes
bajos y desacuerdo entre clasificadores — comportamiento esperado al extrapolar fuera del dominio de entrenamiento.